# 04 — Evaluation

Load best saved model, run full evaluation suite, Grad-CAM visualizations, and error analysis.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path

import config
from data_loader import load_ptbxl, load_mitbih, DatasetSplitter, ECGDataset
from preprocessor import Preprocessor
from evaluate import (
    evaluate_model, compute_metrics, plot_confusion_matrix,
    plot_roc_curves, plot_pr_curves, plot_training_curves,
    grad_cam_1d, compare_models,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load dataset and best checkpoint

In [ ]:
import tensorflow as tf

DATASET = 'ptbxl'   # change to 'mitbih' as needed

try:
    if DATASET == 'ptbxl':
        ds = load_ptbxl(config.PATHS.ptbxl)
        task = 'multilabel'
    else:
        ds = load_mitbih(config.PATHS.mitbih)
        task = 'multiclass'
    print(ds)
except FileNotFoundError as e:
    print(f'Dataset not found: {e}')
    ds = None

# Find best checkpoint
checkpoints = sorted(config.PATHS.models.glob('*.keras'))
if checkpoints:
    CKPT_PATH = str(checkpoints[-1])
    print(f'Loading checkpoint: {CKPT_PATH}')
    model = tf.keras.models.load_model(CKPT_PATH)
    model.summary()
else:
    print('No checkpoints found. Run notebook 03 first.')
    model = None

## 2. Prepare test split

In [ ]:
if ds is not None and model is not None:
    splitter = DatasetSplitter(seed=42)
    _, _, test_ds = splitter.split(ds)

    prep = Preprocessor(fs=test_ds.fs, target_fs=config.SIGNAL.target_fs)
    prep.fit(ds.X)  # fit on full dataset statistics
    X_test = prep.transform(test_ds.X)
    y_test = test_ds.y

    print(f'Test set: {X_test.shape}, y: {y_test.shape}')
else:
    print('Skipping — dataset or model not available.')

## 3. Full evaluation suite

In [ ]:
if ds is not None and model is not None:
    metrics = evaluate_model(
        model, X_test, y_test, ds.labels,
        results_dir=config.PATHS.results,
        n_bootstrap=200,  # use 1000 for full CI
    )

    import pandas as pd
    metrics_df = pd.DataFrame.from_dict(metrics, orient='index', columns=['value'])
    display(metrics_df.style.format('{:.4f}').background_gradient(cmap='Blues'))
else:
    print('Skipping.')

## 4. Confusion matrix

In [ ]:
if ds is not None and model is not None:
    y_score = model.predict(X_test, verbose=0)
    is_multilabel = y_test.ndim == 2

    if is_multilabel:
        y_pred = np.argmax(y_score, axis=1)
        y_true = np.argmax(y_test, axis=1)
    else:
        y_pred = np.argmax(y_score, axis=1)
        y_true = y_test.astype(int)

    fig = plot_confusion_matrix(y_true, y_pred, ds.labels)
    plt.show()

## 5. ROC & PR curves

In [ ]:
if ds is not None and model is not None:
    y_true_bin = y_test if is_multilabel else np.eye(len(ds.labels))[y_true]
    fig_roc = plot_roc_curves(y_true_bin, y_score, ds.labels)
    plt.show()
    fig_pr = plot_pr_curves(y_true_bin, y_score, ds.labels)
    plt.show()

## 6. Grad-CAM on 10 test samples

In [ ]:
if ds is not None and model is not None:
    n_gradcam = 10
    rng = np.random.default_rng(0)
    sel_idx = rng.choice(len(X_test), n_gradcam, replace=False)

    LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']

    for i, idx in enumerate(sel_idx):
        sample = X_test[idx]  # (n_leads, n_t)
        try:
            saliency = grad_cam_1d(model, sample)
        except Exception as e:
            print(f'Grad-CAM failed for sample {idx}: {e}')
            continue

        true_cls = int(y_true[idx]) if not is_multilabel else int(np.argmax(y_test[idx]))
        pred_cls = int(y_pred[idx])
        lead_names = LEAD_NAMES[:sample.shape[0]]

        fig, axes = plt.subplots(2, 1, figsize=(14, 3), sharex=True)
        axes[0].plot(sample[0], lw=0.8, color='steelblue')
        axes[0].set_title(f'Sample {idx}  true={ds.labels[true_cls]}  pred={ds.labels[pred_cls]}')
        axes[0].set_ylabel('Lead I')

        t = np.arange(len(saliency))
        axes[1].fill_between(t, saliency, alpha=0.7, color='red')
        axes[1].set_ylabel('Saliency')
        axes[1].set_xlabel('Time (samples)')
        plt.tight_layout()
        plt.show()

## 7. Error analysis — most confidently wrong predictions

In [ ]:
if ds is not None and model is not None:
    wrong_mask = y_pred != y_true
    wrong_idx = np.where(wrong_mask)[0]
    if len(wrong_idx) == 0:
        print('No wrong predictions!')
    else:
        # Confidence of the wrong predicted class
        wrong_confidence = y_score[wrong_idx, y_pred[wrong_idx]]
        top_wrong = wrong_idx[np.argsort(-wrong_confidence)[:10]]

        import pandas as pd
        rows = []
        for idx in top_wrong:
            rows.append({
                'sample_idx': int(idx),
                'true_class': ds.labels[y_true[idx]],
                'pred_class': ds.labels[y_pred[idx]],
                'confidence': float(y_score[idx, y_pred[idx]]),
            })
        error_df = pd.DataFrame(rows)
        display(error_df)

        # Plot the most confidently wrong sample
        worst_idx = top_wrong[0]
        fig, ax = plt.subplots(figsize=(12, 2))
        ax.plot(X_test[worst_idx, 0], lw=0.8)
        ax.set_title(
            f'Most confidently wrong — true: {ds.labels[y_true[worst_idx]]}'
            f'  pred: {ds.labels[y_pred[worst_idx]]}'
            f'  conf: {y_score[worst_idx, y_pred[worst_idx]]:.1%}'
        )
        plt.tight_layout()
        plt.show()

## 8. MLflow run comparison

In [ ]:
df_runs = compare_models()
if not df_runs.empty:
    display(df_runs.head(20))
else:
    print('No MLflow runs recorded yet.')